In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import xgboost
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
warnings.filterwarnings('ignore')

In [79]:
# Load the training data
df = pd.read_csv('../playground-series-s5e11/train.csv')

print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Dataset shape: (593994, 13)
Number of rows: 593994
Number of columns: 13


## Feature Engineering Section
Add your engineered features below. The analysis functions will automatically include them.

In [ ]:
# Create a copy for feature engineering
df_features = df.copy()

# Example engineered features (replace/add your own):
# df_features['income_to_loan_ratio'] = df_features['annual_income'] / (df_features['loan_amount'] + 1)
# df_features['debt_burden'] = df_features['debt_to_income_ratio'] * df_features['annual_income']
# Add more engineered features here...

print(f"Dataset shape after feature engineering: {df_features.shape}")

## Feature Importance Analysis

In [ ]:
def prepare_features_for_analysis(df, target_col='loan_paid_back'):
    """Prepare features by encoding categorical variables"""
    df_encoded = df.copy()
    
    # Identify categorical columns (exclude id and target)
    cat_cols = df_encoded.select_dtypes(include=['object']).columns.tolist()
    
    # Label encode categorical columns
    le_dict = {}
    for col in cat_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        le_dict[col] = le
    
    # Separate features and target
    X = df_encoded.drop([target_col, 'id'], axis=1, errors='ignore')
    y = df_encoded[target_col]
    
    return X, y, le_dict

X, y, label_encoders = prepare_features_for_analysis(df_features)
print(f"Features for analysis: {X.shape[1]}")
print(f"Feature names: {list(X.columns)}")

In [ ]:
def calculate_feature_importance(X, y, method='all'):
    """
    Calculate feature importance using multiple methods
    
    Parameters:
    - method: 'correlation', 'random_forest', 'xgboost', or 'all'
    """
    results = {}
    
    # Correlation-based importance
    if method in ['correlation', 'all']:
        correlations = X.corrwith(y).abs().sort_values(ascending=False)
        results['correlation'] = correlations
    
    # Random Forest importance
    if method in ['random_forest', 'all']:
        rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X, y)
        rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
        results['random_forest'] = rf_importance
    
    # XGBoost importance
    if method in ['xgboost', 'all']:
        xgb = xgboost.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        xgb.fit(X, y)
        xgb_importance = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
        results['xgboost'] = xgb_importance
    
    return results

# Calculate feature importance
importance_results = calculate_feature_importance(X, y, method='all')

# Display results
for method, importance in importance_results.items():
    print(f"\n{'='*60}")
    print(f"Feature Importance - {method.upper()}")
    print(f"{'='*60}")
    print(importance)

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (method, importance) in enumerate(importance_results.items()):
    top_n = min(15, len(importance))
    importance.head(top_n).plot(kind='barh', ax=axes[idx])
    axes[idx].set_title(f'Top {top_n} Features - {method.upper()}')
    axes[idx].set_xlabel('Importance Score')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

# Create combined importance dataframe
importance_df = pd.DataFrame({
    'Correlation': importance_results['correlation'],
    'Random Forest': importance_results['random_forest'],
    'XGBoost': importance_results['xgboost']
})
importance_df['Mean Importance'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Mean Importance', ascending=False)

print("\nCombined Feature Importance Rankings:")
print(importance_df)

## Multicollinearity Analysis

In [ ]:
def calculate_vif(X, threshold=10):
    """Calculate Variance Inflation Factor for all features"""
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif_data = vif_data.sort_values('VIF', ascending=False)
    
    print(f"\nVariance Inflation Factor (VIF)")
    print(f"{'='*60}")
    print(f"VIF > {threshold} indicates high multicollinearity")
    print(f"{'='*60}")
    print(vif_data)
    
    high_vif = vif_data[vif_data['VIF'] > threshold]
    if len(high_vif) > 0:
        print(f"\n⚠️ Features with high multicollinearity (VIF > {threshold}):")
        print(high_vif)
    
    return vif_data

vif_results = calculate_vif(X, threshold=10)

In [ ]:
# Correlation matrix for all features
correlation_matrix = X.corr()

# Find highly correlated feature pairs
def find_high_correlations(corr_matrix, threshold=0.8):
    """Find pairs of features with correlation above threshold"""
    high_corr_pairs = []
    
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                high_corr_pairs.append({
                    'Feature 1': corr_matrix.columns[i],
                    'Feature 2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    return pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False, key=abs)

high_corr = find_high_correlations(correlation_matrix, threshold=0.8)
print(f"\nHighly Correlated Feature Pairs (|correlation| > 0.8):")
print(f"{'='*60}")
if len(high_corr) > 0:
    print(high_corr)
else:
    print("No feature pairs with correlation > 0.8")

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix (All Features)', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

## Feature Selection Recommendations

In [ ]:
def recommend_features_to_drop(importance_df, vif_results, high_corr, 
                                importance_threshold=0.01, vif_threshold=10):
    """
    Recommend features to drop based on:
    1. Low importance across all methods
    2. High multicollinearity (VIF)
    3. High correlation with other features
    """
    recommendations = {
        'low_importance': [],
        'high_vif': [],
        'high_correlation': []
    }
    
    # Low importance features
    low_importance = importance_df[importance_df['Mean Importance'] < importance_threshold].index.tolist()
    recommendations['low_importance'] = low_importance
    
    # High VIF features
    high_vif = vif_results[vif_results['VIF'] > vif_threshold]['Feature'].tolist()
    recommendations['high_vif'] = high_vif
    
    # Features in highly correlated pairs (recommend dropping the less important one)
    if len(high_corr) > 0:
        for _, row in high_corr.iterrows():
            feat1, feat2 = row['Feature 1'], row['Feature 2']
            # Drop the one with lower mean importance
            if importance_df.loc[feat1, 'Mean Importance'] < importance_df.loc[feat2, 'Mean Importance']:
                recommendations['high_correlation'].append(feat1)
            else:
                recommendations['high_correlation'].append(feat2)
    
    recommendations['high_correlation'] = list(set(recommendations['high_correlation']))
    
    print("Feature Dropping Recommendations:")
    print(f"{'='*60}")
    print(f"\n1. Low Importance Features (Mean Importance < {importance_threshold}):")
    print(f"   {recommendations['low_importance']}")
    print(f"\n2. High Multicollinearity Features (VIF > {vif_threshold}):")
    print(f"   {recommendations['high_vif']}")
    print(f"\n3. Features in Highly Correlated Pairs:")
    print(f"   {recommendations['high_correlation']}")
    
    all_recommendations = set(recommendations['low_importance'] + 
                             recommendations['high_vif'] + 
                             recommendations['high_correlation'])
    print(f"\n{'='*60}")
    print(f"All Recommended Features to Consider Dropping:")
    print(f"{sorted(all_recommendations)}")
    
    return recommendations

recommendations = recommend_features_to_drop(importance_df, vif_results, high_corr,
                                            importance_threshold=0.01, vif_threshold=10)